In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report,roc_auc_score,accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV,StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from scikeras.wrappers import KerasClassifier

In [2]:
df=pd.read_csv('processed.csv')
print('Data shape:',df.shape)

Data shape: (1069, 22)


In [3]:
list(enumerate(df.columns))

[(0, 'w/b'),
 (1, 'Water'),
 (2, 'Cement type'),
 (3, 'Cement'),
 (4, 'Slag'),
 (5, 'Fly ash'),
 (6, 'Silica fume'),
 (7, 'Lime filler'),
 (8, 'FA'),
 (9, 'CA'),
 (10, 'Plasticizer'),
 (11, 'Superplasticizer'),
 (12, 'Air entraining'),
 (13, 'Comp. str. test age'),
 (14, 'Compressive strength'),
 (15, 'Air content'),
 (16, 'Spreed'),
 (17, 'Slump'),
 (18, 'Fresh density'),
 (19, 'Dry  density'),
 (20, 'Migration test age'),
 (21, 'Migration resistance')]

1-hot encoding

In [4]:
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [2])], remainder='passthrough')
X=ct.fit_transform(df)
# remove dummy variable and output
y=X[:,-1]
X=X[:,1:-1]
print(X.shape)
print(y.shape)

(1069, 30)
(1069,)


ANN

In [5]:
def get_ann(optimizer='rmsprop', init='glorot_uniform'):
    ann=keras.models.Sequential()
    ann.add(keras.layers.Dense(60,activation='relu',kernel_initializer=init,
                            input_shape=(X.shape[1],)))
    ann.add(keras.layers.Dense(32,activation='relu'))
    ann.add(keras.layers.Dense(32,activation='relu'))
    ann.add(keras.layers.Dense(5,activation='softmax'))
    ann.compile(optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return ann

Grid Search

In [6]:
param_grid={
    'optimizer': ['rmsprop', 'adam'],
    'model__init': ['glorot_uniform', 'normal', 'uniform'],
    'epochs': [200,400],
    'batch_size': [16,32,64]
}

splits = list(StratifiedKFold(shuffle=True,random_state=0).split(X,y))
train_index, test_index = splits[0]
X_train, X_test = X[train_index], X[test_index]
y_train, y_test = y[train_index], y[test_index]
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

def train_ann(X_train,y_train):
    grid=GridSearchCV(KerasClassifier(get_ann,verbose=0,random_state=0),
                param_grid,cv=5,n_jobs=3,scoring='accuracy')
    grid.fit(X_train, y_train)
    print(grid.best_params_)
    print(grid.best_score_)
    return grid

In [8]:
grid=train_ann(X_train,y_train)

d:\College files\9th sem\mtp\code\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


{'batch_size': 16, 'epochs': 400, 'model__init': 'glorot_uniform', 'optimizer': 'rmsprop'}
0.8280701754385964


In [7]:
best_params={'batch_size': 16, 'epochs': 400, 'model__init': 'glorot_uniform', 'optimizer': 'rmsprop'}
best_model=KerasClassifier(get_ann,verbose=0,random_state=0,**best_params)
best_model.fit(X_train,y_train)
print(classification_report(y_test,best_model.predict(X_test)))
roc_auc_score(y_test,best_model.predict_proba(X_test),multi_class='ovr')

              precision    recall  f1-score   support

         0.0       0.98      0.95      0.96        43
         1.0       0.80      0.86      0.83        42
         2.0       0.78      0.74      0.76        43
         3.0       0.84      0.86      0.85        43
         4.0       0.98      0.95      0.96        43

    accuracy                           0.87       214
   macro avg       0.87      0.87      0.87       214
weighted avg       0.88      0.87      0.87       214



0.9678649142235434

Removing cement type

In [8]:
X=df.drop('Cement type',axis=1).iloc[:,:-1].values
y=df.iloc[:,-1].values
print(X.shape)
print(y.shape)

(1069, 20)
(1069,)


In [9]:
X_train, X_test = X[train_index], X[test_index]
y_train, y_test = y[train_index], y[test_index]
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
grid=train_ann(X_train,y_train)

d:\College files\9th sem\mtp\code\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


{'batch_size': 32, 'epochs': 400, 'model__init': 'uniform', 'optimizer': 'rmsprop'}
0.8467836257309942


In [10]:
best_params={'batch_size': 32, 'epochs': 400, 'model__init': 'uniform', 'optimizer': 'rmsprop'}
best_model=KerasClassifier(get_ann,verbose=0,random_state=0,**best_params)
best_model.fit(X_train,y_train)
print(classification_report(y_test,best_model.predict(X_test)))
roc_auc_score(y_test,best_model.predict_proba(X_test),multi_class='ovr')

              precision    recall  f1-score   support

           0       0.93      1.00      0.97        43
           1       0.90      0.88      0.89        42
           2       0.79      0.79      0.79        43
           3       0.86      0.86      0.86        43
           4       1.00      0.95      0.98        43

    accuracy                           0.90       214
   macro avg       0.90      0.90      0.90       214
weighted avg       0.90      0.90      0.90       214



0.9767315575761109

In [12]:
accuracy_score(y_test,best_model.predict(X_test))

0.897196261682243